In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

os.makedirs('figures', exist_ok=True)

plt.rcParams.update({
    'font.size': 14, 'axes.titlesize': 15, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 12,
})

# ── Load data ─────────────────────────────────────────────────────────────────
df  = pd.read_csv('Results/paper.csv')   # SNAIL (old schema: config, wraparound)
dft = pd.read_csv('Results/ibm.csv')     # IBM Brisbane (new schema: router, alpha, beta)

df['config']  = df['config'].replace({'Standard SABRE': 'SABRE', 'Standard MIRAGE': 'MIRAGE'})
dft['config'] = dft['router']            # normalise to 'config'

df['topo']  = df['device'] + df['wraparound'].map({True: '_wrap', False: ''})
dft['topo'] = dft['device']

# ── Constants ─────────────────────────────────────────────────────────────────
CONFIG_ORDER = ['SABRE', 'FASST', 'MIRAGE', 'FINESSE']
CFG_COLORS   = ['#4C72B0', '#1B6B3A', '#DD8452', '#7B1D1D']

TOPO_ORDER = [
    'square_ring', 'square_ring_wrap', 'square_ring_diag', 'square_ring_diag_wrap',
    'square_ring_full', 'square_ring_full_wrap', 'pentagon_ring', 'pentagon_ring_wrap',
]
TOPO_LABELS = {
    'square_ring':           '4q/4e module',
    'square_ring_diag':      '4q/5e module',
    'square_ring_full':      '4q/6e module',
    'pentagon_ring':         '5q/7e module',
}

CIRCUIT_ORDER = [
    'qpeexact_n8', 'wstate_n8', 'qft_n10', 'ae_n10', 'ghz_n10', 'vqe_two_local_n10',
    'seca_n11', 'multiplier_n15', 'dnn_n16', 'qec9xz_n17', 'square_root_n18', 'bv_n19',
    'qft_n24', 'qaoa_n25_p3', 'ising_n26', 'qft_n32', 'qaoa_n32_p3', 'random_n32_d50',
]
CIRC_LABELS = {
    'qpeexact_n8': 'QPE\n(n8)', 'wstate_n8': 'W-st\n(n8)', 'qft_n10': 'QFT\n(n10)',
    'ae_n10': 'AE\n(n10)', 'ghz_n10': 'GHZ\n(n10)', 'vqe_two_local_n10': 'VQE\n(n10)',
    'seca_n11': 'SECA\n(n11)', 'multiplier_n15': 'Mult\n(n15)', 'dnn_n16': 'DNN\n(n16)',
    'qec9xz_n17': 'QEC\n(n17)', 'square_root_n18': '√n\n(n18)', 'bv_n19': 'BV\n(n19)',
    'qft_n24': 'QFT\n(n24)', 'qaoa_n25_p3': 'QAOA\n(n25)', 'ising_n26': 'Ising\n(n26)',
    'qft_n32': 'QFT\n(n32)', 'qaoa_n32_p3': 'QAOA\n(n32)', 'random_n32_d50': 'Rand\n(n32)',
}

POST_NATIVE = {'SABRE': 'swaps', 'MIRAGE': 'depth', 'FASST': 'lf_cost', 'FINESSE': 'lf_cost'}
POST_LF     = {c: 'lf_cost' for c in CONFIG_ORDER}

def post_select(df, post_map, groupby=('topo', 'circuit')):
    frames = []
    for cfg, col in post_map.items():
        sub = df[df.config == cfg]
        if sub.empty: continue
        frames.append(sub.loc[sub.groupby(list(groupby))[col].idxmin()])
    return pd.concat(frames).reset_index(drop=True)

# ── Build filtered sets ────────────────────────────────────────────────────────
present_topos = [t for t in TOPO_ORDER if t in df['topo'].values]
nowrap_topos  = [t for t in present_topos if not t.endswith('_wrap')]
present_circs = [c for c in CIRCUIT_ORDER if c in df['circuit'].values]

_snail = df[~df['wraparound'] & df['topo'].isin(nowrap_topos)]
_ibm   = dft[dft['topo'] == 'fake_brisbane']

ps_snail_nat = post_select(_snail, POST_NATIVE)
ps_snail_lf  = post_select(_snail, POST_LF)
ps_ibm_nat   = post_select(_ibm,   POST_NATIVE, groupby=('topo', 'circuit'))
ps_ibm_lf    = post_select(_ibm,   POST_LF,     groupby=('topo', 'circuit'))

ibm_circs = [c for c in CIRCUIT_ORDER if c in _ibm['circuit'].values]

PS_METHODS = [
    ('native', ps_snail_nat, ps_ibm_nat, 'Native PS  (SABRE→swaps, MIRAGE→depth, FASST/FINESSE→lf)'),
    ('lf',     ps_snail_lf,  ps_ibm_lf,  'LF PS  (all configs→min lf_cost)'),
]

print(f'SNAIL topologies (non-wrapped): {nowrap_topos}')
print(f'Circuits: {len(present_circs)}, IBM circuits: {len(ibm_circs)}')

In [ ]:
# ── Beta grid search: justifying β=1 default ──────────────────────────────────
# FINESSE uses D' = α·D_hop + β·D_fid. We fix α=1 and sweep β.
# Equivalence: old fidelity_blend=0.5 → 0.5·D_hop + 0.5·D_fid, same ratio as β=1 → no rerun needed.

dg  = pd.read_csv('Results/grid_snail.csv')
dgi = pd.read_csv('Results/grid_ibm.csv')
dgi = dgi[dgi.device == 'fake_brisbane']  # Brisbane only

BETA_ORDER = [0.0, 1.0, 5.0, 10.0, 20.0, 33.0, 100.0]

def grid_summary(df, label):
    frames_nat, frames_lf = [], []
    for keys, sub in df.groupby(['device','circuit','router','alpha','beta'], dropna=False):
        col = 'swaps' if keys[2] == 'SABRE' else 'lf_cost'
        frames_nat.append(sub.nsmallest(1, col))
        frames_lf.append(sub.nsmallest(1, 'lf_cost'))
    ps_nat = pd.concat(frames_nat)
    ps_lf  = pd.concat(frames_lf)
    sabre_nat = ps_nat[ps_nat.router=='SABRE'].set_index(['device','circuit'])['lf_cost']
    sabre_lf  = ps_lf[ps_lf.router =='SABRE'].set_index(['device','circuit'])['lf_cost']
    print(f'=== {label} ===')
    print(f'  {"β":>6}  {"native ps":>10}  {"lf ps":>10}')
    print('  ' + '-'*32)
    for b in BETA_ORDER:
        m = ps_nat[(ps_nat.router=='FINESSE')&(ps_nat.alpha==1.0)&(ps_nat.beta==b)].set_index(['device','circuit'])['lf_cost']
        l = ps_lf[ (ps_lf.router =='FINESSE')&(ps_lf.alpha ==1.0)&(ps_lf.beta ==b)].set_index(['device','circuit'])['lf_cost']
        c = sabre_nat.index.intersection(m.index)
        if c.empty: continue
        pn = 100*(m[c]-sabre_nat[c])/sabre_nat[c]
        pl = 100*(l[c]-sabre_lf[c])/sabre_lf[c]
        marker = '  ← chosen default (equiv. fidelity_blend=0.5)' if b == 1.0 else ''
        print(f'  {b:>6.0f}  {pn.mean():>+9.1f}%  {pl.mean():>+9.1f}%{marker}')
    m = ps_nat[(ps_nat.router=='FINESSE')&(ps_nat.alpha==0.0)&(ps_nat.beta==1.0)].set_index(['device','circuit'])['lf_cost']
    l = ps_lf[ (ps_lf.router =='FINESSE')&(ps_lf.alpha ==0.0)&(ps_lf.beta ==1.0)].set_index(['device','circuit'])['lf_cost']
    c = sabre_nat.index.intersection(m.index)
    if not c.empty:
        pn = 100*(m[c]-sabre_nat[c])/sabre_nat[c]
        pl = 100*(l[c]-sabre_lf[c])/sabre_lf[c]
        print(f'  {"α=0":>6}  {pn.mean():>+9.1f}%  {pl.mean():>+9.1f}%  ← pure fidelity')
    print()

grid_summary(dg,  'SNAIL (24 seeds, 4 topologies, 6 circuits)')
grid_summary(dgi, 'IBM Brisbane (24 seeds, 6 circuits)')
print('β=1 is best or near-best on both platforms. High β hurts pentagon_ring and IBM.')
print('Pure fidelity (α=0) fails on large circuits — loses hop-count direction.')

In [ ]:
# ── Per-circuit absolute values — SNAIL averaged across topologies ─────────────
# Two figures (lf_cost, depth), each with two subplots (native ps, lf ps).

circs = [c for c in CIRCUIT_ORDER if c in ps_snail_nat['circuit'].values]
xs    = np.arange(len(circs))
n_cfg = len(CONFIG_ORDER)
w     = 0.80 / n_cfg

METRICS = [
    ('lf_cost', r'$\Sigma\,{-}\log F$', 'fig_v2_snail_lf_per_circ.pdf'),
    ('depth',   'Depth',                 'fig_v2_snail_depth_per_circ.pdf'),
]

for col, ylabel, fname in METRICS:
    fig, axes = plt.subplots(2, 1, figsize=(max(13, len(circs)*1.3), 11), sharey=False)
    for ax, (ps_key, ps_snail, ps_ibm, ps_title) in zip(axes, PS_METHODS):
        ax.set_title(ps_title, fontweight='bold')
        for k, (cfg, color) in enumerate(zip(CONFIG_ORDER, CFG_COLORS)):
            vals = []
            for circ in circs:
                topo_vals = [
                    float(ps_snail[(ps_snail.topo==t)&(ps_snail.circuit==circ)&(ps_snail.config==cfg)][col].values[0])
                    for t in nowrap_topos
                    if len(ps_snail[(ps_snail.topo==t)&(ps_snail.circuit==circ)&(ps_snail.config==cfg)])
                ]
                vals.append(np.mean(topo_vals) if topo_vals else np.nan)
            ax.bar(xs + (k - (n_cfg-1)/2)*w, vals, w, color=color, label=cfg)
        ax.set_yscale('log')
        ax.yaxis.set_major_formatter(mticker.LogFormatterSciNotation(labelOnlyBase=False))
        ax.yaxis.set_minor_locator(mticker.LogLocator(subs='auto'))
        ax.set_xticks(xs); ax.set_xticklabels([CIRC_LABELS.get(c, c) for c in circs])
        ax.set_ylabel(ylabel)
        ax.legend(frameon=False, ncol=4, loc='upper left')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.suptitle(f'{ylabel} — SNAIL (averaged across 4 topologies)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'figures/{fname}', bbox_inches='tight')
    plt.show()
    print(f'Saved → figures/{fname}')

In [ ]:
# ── SNAIL by-topology: lf_cost + depth, both ps methods ──────────────────────
# For each metric: figure with rows=ps method, cols=topology.

METRICS = [
    ('lf_cost', r'$\Sigma\,{-}\log F$', 'fig_v2_snail_lf_by_topo.pdf'),
    ('depth',   'Depth',                 'fig_v2_snail_depth_by_topo.pdf'),
]

n_topo = len(nowrap_topos)
n_ps   = 2
n_cfg  = len(CONFIG_ORDER)
w      = 0.80 / n_cfg

for col, ylabel, fname in METRICS:
    fig, axes = plt.subplots(n_ps, n_topo,
                             figsize=(5.5*n_topo, 6*n_ps), sharey=False)
    for row, (ps_key, ps_snail, ps_ibm, ps_title) in enumerate(PS_METHODS):
        for col_idx, topo in enumerate(nowrap_topos):
            ax = axes[row, col_idx]
            topo_circs = [c for c in CIRCUIT_ORDER
                          if len(ps_snail[(ps_snail.topo==topo)&(ps_snail.circuit==c)])]
            xs = np.arange(len(topo_circs))
            ax.set_title(f'{TOPO_LABELS.get(topo, topo)}\n{ps_title}', fontsize=11, fontweight='bold')
            for k, (cfg, color) in enumerate(zip(CONFIG_ORDER, CFG_COLORS)):
                vals = [
                    float(ps_snail[(ps_snail.topo==topo)&(ps_snail.circuit==c)&(ps_snail.config==cfg)][col].values[0])
                    if len(ps_snail[(ps_snail.topo==topo)&(ps_snail.circuit==c)&(ps_snail.config==cfg)]) else np.nan
                    for c in topo_circs
                ]
                ax.bar(xs + (k - (n_cfg-1)/2)*w, vals, w, color=color, label=cfg)
            ax.set_yscale('log')
            ax.yaxis.set_major_formatter(mticker.LogFormatterSciNotation(labelOnlyBase=False))
            ax.yaxis.set_minor_locator(mticker.LogLocator(subs='auto'))
            ax.set_xticks(xs)
            ax.set_xticklabels([CIRC_LABELS.get(c, c) for c in topo_circs], fontsize=9)
            ax.set_ylabel(ylabel)
            if row == 0 and col_idx == 0:
                ax.legend(frameon=False, ncol=2, fontsize=9)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.suptitle(f'{ylabel} — SNAIL by topology', fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'figures/{fname}', bbox_inches='tight')
    plt.show()
    print(f'Saved → figures/{fname}')

In [ ]:
# ── IBM Brisbane: lf_cost + depth, both ps methods ───────────────────────────
# Same format as SNAIL per-circuit but for a single backend.

circs_ibm = [c for c in CIRCUIT_ORDER if c in ps_ibm_nat['circuit'].values]
xs    = np.arange(len(circs_ibm))
n_cfg = len(CONFIG_ORDER)
w     = 0.80 / n_cfg

METRICS = [
    ('lf_cost', r'$\Sigma\,{-}\log F$', 'fig_v2_ibm_lf.pdf'),
    ('depth',   'Depth',                 'fig_v2_ibm_depth.pdf'),
]

for col, ylabel, fname in METRICS:
    fig, axes = plt.subplots(2, 1, figsize=(max(13, len(circs_ibm)*1.3), 11), sharey=False)
    for ax, (ps_key, ps_snail, ps_ibm, ps_title) in zip(axes, PS_METHODS):
        ax.set_title(ps_title, fontweight='bold')
        for k, (cfg, color) in enumerate(zip(CONFIG_ORDER, CFG_COLORS)):
            vals = [
                float(ps_ibm[(ps_ibm.circuit==c)&(ps_ibm.config==cfg)][col].values[0])
                if len(ps_ibm[(ps_ibm.circuit==c)&(ps_ibm.config==cfg)]) else np.nan
                for c in circs_ibm
            ]
            ax.bar(xs + (k - (n_cfg-1)/2)*w, vals, w, color=color, label=cfg)
        ax.set_yscale('log')
        ax.yaxis.set_major_formatter(mticker.LogFormatterSciNotation(labelOnlyBase=False))
        ax.yaxis.set_minor_locator(mticker.LogLocator(subs='auto'))
        ax.set_xticks(xs); ax.set_xticklabels([CIRC_LABELS.get(c, c) for c in circs_ibm])
        ax.set_ylabel(ylabel)
        ax.legend(frameon=False, ncol=4, loc='upper left')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.suptitle(f'{ylabel} — IBM Brisbane (127q)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'figures/{fname}', bbox_inches='tight')
    plt.show()
    print(f'Saved → figures/{fname}')